# Benchmark the SFT model vs other models — Open LLM Leaderboard v1

`sparky_leaderboard.py` runs the standard suite — ARC-Challenge (25-shot),
HellaSwag (10-shot), MMLU (5-shot), TruthfulQA-MC2 (0-shot), Winogrande (5-shot),
GSM8K (5-shot) — each with its canonical few-shot/metric via `lm_eval`, and an
**Average** (leaderboard convention). These numbers are directly comparable to
other models' published Open LLM Leaderboard v1 results.

**Targets the SFT checkpoint by default** (`sft_epoch2.pth`). Note: the v1 harness
scores all models in **raw few-shot format (no chat template)** — that's exactly
how other instruct/chat models were measured there, so the comparison is fair.
Run the BASE checkpoint too if you want to see the SFT delta.

Needs a **GPU**. A **full** run is slow (tens of minutes to hours).

In [ ]:
# Mount Drive + paths
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os
SYNAPSE_DIR = '/content/drive/MyDrive/synapse'
SFT_CKPT  = f'{SYNAPSE_DIR}/sft_checkpoints/v3_15source/sft_best.pth'  # <-- benchmarking THIS (v3)
BASE_CKPT = f'{SYNAPSE_DIR}/checkpoints/synapse_2b_d2560_l28.pth'   # alt: the pretrain base
CKPT      = SFT_CKPT                                                # switch to BASE_CKPT to compare
TOKENIZER = f'{SYNAPSE_DIR}/tokenizer_out/tokenizer.json'
tag = 'sft' if CKPT == SFT_CKPT else 'base'
RESULTS   = f'{SYNAPSE_DIR}/leaderboard_results/synapse_2b_{tag}.json'
print('CKPT =', CKPT)
print('RESULTS =', RESULTS)

In [ ]:
# Clone/pull repo
import subprocess
REPO_DIR='/content/synapse_repo'
if os.path.isdir(os.path.join(REPO_DIR,'.git')):
    subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'],check=True)
else:
    subprocess.run(['git','clone','--depth=1','https://github.com/ajencinas/synapse.git',REPO_DIR],check=True)
assert os.path.isfile(os.path.join(REPO_DIR,'sparky','sparky_leaderboard.py'))
print('REPO_DIR =', REPO_DIR)

In [ ]:
# Deps (lm_eval pulls datasets etc.) + GPU check
!pip install -q lm_eval tokenizers
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
# Preflight
for label,p in [('checkpoint',CKPT),('tokenizer',TOKENIZER)]:
    print(('  OK ' if os.path.exists(p) else 'MISS ')+f'{label}: {p}')
assert os.path.exists(CKPT) and os.path.exists(TOKENIZER), 'missing inputs'
os.makedirs(os.path.dirname(RESULTS), exist_ok=True)

In [ ]:
# LIMIT=50 = FAST, NON-comparable smoke check. Set LIMIT=None for the FULL run.
# --no-compile: lm_eval feeds many input lengths; compile's CUDA-graph mode would
# re-record per length ('distinct sizes' warning) and run SLOWER here.
LIMIT = 50
TASKS = ''            # '' = all six; or e.g. 'mmlu,gsm8k'
limit_flag = f'--limit {LIMIT}' if LIMIT else ''
tasks_flag = f'--tasks {TASKS}' if TASKS else ''
cmd = (f'cd {REPO_DIR}/sparky && python sparky_leaderboard.py '
       f'--ckpt "{CKPT}" --tokenizer "{TOKENIZER}" --no-compile '
       f'--output "{RESULTS}" {limit_flag} {tasks_flag}')
print(cmd)
!{cmd}

In [ ]:
# Show results
import json
if os.path.exists(RESULTS):
    print(json.dumps(json.load(open(RESULTS)), indent=2)[:3000])
else:
    print('no results at', RESULTS)

## Notes
- **Comparable numbers require the FULL run** (`LIMIT = None`). `--limit` is a fast
  sanity estimate only — never report it as a leaderboard score.
- **Always report the `lm_eval` version** printed during the run (scores shift
  slightly across harness versions).
- **SFT delta**: run once with `CKPT = SFT_CKPT` and once with `CKPT = BASE_CKPT`
  to see what SFT changed (expect gains on GSM8K/MMLU-style tasks; small movement
  elsewhere). Both are saved as `synapse_2b_sft.json` / `synapse_2b_base.json`.
- For chat-quality (not leaderboard) testing, use `sft_bench_colab.ipynb`.